# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 5: Fine-tuning a Frontier Model

Now we will use OpenAI's API to fine-tune our own private variant of GPT-4.1-nano

In [1]:
# imports

import os
import re
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items  import Item
from pricer.evaluator import evaluate

In [2]:
# environment

LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
username = "khirodsahoo93"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [4]:
openai = OpenAI()

# Data size

OpenAI recommends fine-tuning with a small population of 50-100 examples

I'm going to go with 20,000 points.

This cost me $3.42 - you should stick with 100 examples and the cost will be minimal!

In [5]:
# OpenAI recommends fine-tuning with populations of 50-100 examples
# But as our examples are very small, I'm suggesting we go with 100 examples (and 1 epoch)


fine_tune_train = train[:100]
fine_tune_validation = val[:50]

In [6]:
len(fine_tune_train)

100

# Step 1

Prepare our data for fine-tuning in JSONL (JSON Lines) format and upload to OpenAI

In [7]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [8]:
messages_for(fine_tune_train[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Odyssey Krom Utility/Record Case – Black  \nCategory: Music & Audio Accessories  \nBrand: Odyssey  \nDescription: A robust black utility case designed to securely hold 120 7" vinyl records in two compartments.  \nDetails: Features chrome‑plated hardware, a removable lid, fully foam‑lined interior, rubber feet, stacking lid, and a heavy‑duty latch with spring‑loaded handle.'},
 {'role': 'assistant', 'content': '$119.95'}]

In [9]:
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [10]:
print(make_jsonl(train[:3]))

{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Odyssey Krom Utility/Record Case \u2013 Black  \nCategory: Music & Audio Accessories  \nBrand: Odyssey  \nDescription: A robust black utility case designed to securely hold 120 7\" vinyl records in two compartments.  \nDetails: Features chrome\u2011plated hardware, a removable lid, fully foam\u2011lined interior, rubber feet, stacking lid, and a heavy\u2011duty latch with spring\u2011loaded handle."}, {"role": "assistant", "content": "$119.95"}]}
{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: HP 15t-dw300 FHD IPS Laptop  \nCategory: Electronics  \nBrand: HP  \nDescription: A 15.6\" full\u2011HD IPS laptop powered by an 11th\u2011gen Intel i5 processor with 16\u202fGB RAM and a 512\u202fGB PCIe SSD.  \nDetails: Features Intel Iris Xe graphics, Wi\u2011Fi 6, Bluetooth, HDMI, 2\u00d7

In [11]:
# Convert the items into jsonl and write them to a file

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [12]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")

In [13]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [14]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [15]:
train_file

FileObject(id='file-HCPwWoouEJiL8a1kPGMBzc', bytes=56292, created_at=1768876261, filename='fine_tune_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [16]:
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [17]:
validation_file

FileObject(id='file-LVmT8LjuoWRXVUP3DgAPdg', bytes=26442, created_at=1768876264, filename='fine_tune_validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

https://platform.openai.com/storage/files/

# Step 2

## And now time to Fine-tune!

In [18]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer"
)

FineTuningJob(id='ftjob-wk2AxR4FfBU9N5uPQMA7OdWr', created_at=1768876273, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-PZ03WI3snpO0pTJ7vx992T7A', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-HCPwWoouEJiL8a1kPGMBzc', validation_file='file-LVmT8LjuoWRXVUP3DgAPdg', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None)

In [19]:
openai.fine_tuning.jobs.list(limit=1)

SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-wk2AxR4FfBU9N5uPQMA7OdWr', created_at=1768876273, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-PZ03WI3snpO0pTJ7vx992T7A', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-HCPwWoouEJiL8a1kPGMBzc', validation_file='file-LVmT8LjuoWRXVUP3DgAPdg', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None)], has_more=False, object='list')

In [20]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [21]:
job_id

'ftjob-wk2AxR4FfBU9N5uPQMA7OdWr'

In [22]:
openai.fine_tuning.jobs.retrieve(job_id)

FineTuningJob(id='ftjob-wk2AxR4FfBU9N5uPQMA7OdWr', created_at=1768876273, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-PZ03WI3snpO0pTJ7vx992T7A', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-HCPwWoouEJiL8a1kPGMBzc', validation_file='file-LVmT8LjuoWRXVUP3DgAPdg', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None)

In [49]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

[FineTuningJobEvent(id='ftevent-QxATKIU2gEaXr9Yy1YwfHTPl', created_at=1768877560, level='info', message='The job has successfully completed', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-s9bBVHTeZePmTZ60wDG6EVAQ', created_at=1768877559, level='info', message='Usage policy evaluations completed, model is now enabled for sampling', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-u9WD11VJ1CF9OzvOhiEC1G1g', created_at=1768877559, level='info', message='Moderation checks for snapshot ft:gpt-4.1-nano-2025-04-14:personal:pricer:CzvxQaJy passed.', object='fine_tuning.job.event', data={'blocked': False, 'results': [{'flagged': False, 'category': 'harassment/threatening', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual/minors', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'propaganda', 'enforcement': 'blocking

https://platform.openai.com/finetune


# Step 3

Test our fine tuned model

In [50]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [51]:
fine_tuned_model_name

'ft:gpt-4.1-nano-2025-04-14:personal:pricer:CzvxQaJy'

In [52]:
# The prompt

def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
    ]

In [53]:
# Try this out

test_messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Moen 143530 Diverter Plug  \nCategory: Hardware  \nBrand: Moen  \nDescription: A lightweight, unfinished diverter plug for side spray faucets.  \nDetails: Measures 4.25\u202f×\u202f0.78\u202f×\u202f0.96\u202fin., weighs 0.02\u202flb, and is sold as a single unit.'}]

In [54]:
# The inference function


def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

In [55]:
print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))

18.17
$6.74


In [56]:
evaluate(gpt_4__1_nano_fine_tuned, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$7 $1 $40 $3 $6 $65 $65 $8 $10 $2 $21 $0 $4 $14 $413 $10 $49 $6 $54 $11 $11 $9 $40 $136 $3 $112 $84 $16 $85 $94 $15 $100 $123 $396 $85 $134 $21 $97 $117 $12 $216 $111 $61 $205 $1 $9 $223 $634 $136 $62 $20 $34 $1 $27 $8 $54 $23 $464 $7 $3 $29 $66 $60 $123 $20 $99 $5 $69 $573 $27 $4 $23 $47 $0 $0 $42 $7 $15 $1 $8 $210 $183 $45 $22 $41 $48 $0 $63 $74 $12 $43 $54 $133 $44 $27 $73 $14 $361 $4 $104 $125 $16 $18 $1 $54 $74 $140 $40 $17 $266 $11 $22 $5 $3 $19 $15 $301 $5 $73 $34 $57 $24 $610 $48 $30 $74 $35 $8 $258 $516 $55 $57 $79 $10 $16 $33 $103 $128 $20 $63 $91 $788 $328 $9 $9 $33 $11 $69 $3 $231 $35 $139 $54 $357 $26 $97 $154 $5 $16 $18 $22 $78 $8 $13 $20 $14 $20 $23 $56 $55 $207 $160 $7 $26 $49 $89 $240 $44 $181 $11 $42 $25 $12 $117 $61 $20 $35 $169 $67 $22 $67 $1 $16 $92 $10 $6 $2 $67 $246 $9 

In [ ]:
# 96.58 - mini 200
# 79.29 - mini 2000
# 82.26 - nano 2000
# 67.75 - nano 20,000